# R03 · Real-time Linux - periodic loops, latency, jitter and deferred interrupts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R03_real_time_linux_periodic_loops_latency_jitter_an.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 60 min

**Skill Hub concepts:** Drift-free periodic timing; Interrupt latency and jitter; Memory locking and absolute sleeps; SCHED_FIFO and PREEMPT_RT; Deferred interrupt processing

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R03")

## Map: what a 1 kHz control thread on Linux needs

```
 relative sleep:  wake -> work -> sleep(1 ms) -> wake ...      period = 1 ms + work + wakeup latency  -> drifts
 absolute sleep:  next += 1 ms; clock_nanosleep(CLOCK_MONOTONIC, TIMER_ABSTIME, &next)  -> no drift, only jitter
```

| Ingredient | Call | Why |
|---|---|---|
| Absolute deadlines | `clock_nanosleep(..., TIMER_ABSTIME, ...)` | errors do not accumulate |
| Real-time priority | `sched_setscheduler(0, SCHED_FIFO, prio 80)` (needs CAP_SYS_NICE) | runs before every normal task |
| Locked memory | `mlockall(MCL_CURRENT \| MCL_FUTURE)` + prefault the stack | no page faults inside the loop |
| Preemptible kernel | `PREEMPT_RT` (mainline since 6.12) | threaded interrupts, sleeping spinlocks: bounded latency |
| Isolated core | `isolcpus=3 nohz_full=3`, pin the thread (lab R07) | no other work on that core |

**Latency** = actual wake time - intended wake time; **jitter** = its spread (max - min, or a high percentile).
A stock kernel gives tens of microseconds typical with millisecond outliers; PREEMPT_RT bounds the worst case
(tens of microseconds on good hardware).

**Deferred interrupt processing:** keep the interrupt handler tiny (acknowledge, grab data, wake someone) and do the
work later in thread context: Linux threaded IRQs / workqueues / softirqs, FreeRTOS `vTaskNotifyGiveFromISR` to a
high-priority task. Short handlers keep interrupt latency low for everyone else.

## Task 1: a drift-free periodic loop in C
`run_periodic` wakes every `period_ns` for `iters` iterations using absolute sleeps, and stores each wake-up latency
(actual minus intended, nanoseconds) in `lat`. Return the total time from the first intended wake to the last.

In [ ]:
%%writefile periodic.c
#define _GNU_SOURCE
#include <stdint.h>
#include <time.h>

static int64_t ts_ns(const struct timespec *t) { return (int64_t)t->tv_sec * 1000000000LL + t->tv_nsec; }

int64_t run_periodic(int64_t period_ns, int iters, int64_t *lat) {
    struct timespec next, now;
    clock_gettime(CLOCK_MONOTONIC, &next);
    int64_t first = ts_ns(&next) + period_ns, last = first;
    (void)now; (void)iters; (void)lat;   /* TODO: advance next by period_ns (normalize tv_nsec), sleep until it, record latency */
    return last - first;
}

In [ ]:
lab.check("periodic")

Compare with a relative sleep and look at the latency distribution. (Colab is a shared VM with a normal kernel, so
expect large outliers: that is the point.)

In [ ]:
%%writefile compare.c
#define _GNU_SOURCE
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <sched.h>
#include <sys/mman.h>
#include <time.h>
int64_t run_periodic(int64_t period_ns, int iters, int64_t *lat);
static int cmp(const void *a, const void *b) { int64_t x = *(const int64_t *)a, y = *(const int64_t *)b; return (x > y) - (x < y); }
int main(void) {
    enum { N = 2000 };
    static int64_t lat[N];
    printf("mlockall: %s\n", mlockall(MCL_CURRENT | MCL_FUTURE) == 0 ? "ok" : "not permitted");
    struct sched_param sp = { .sched_priority = 80 };
    printf("SCHED_FIFO: %s\n", sched_setscheduler(0, SCHED_FIFO, &sp) == 0 ? "ok" : "not permitted here");
    struct timespec a, b, d = {0, 1000000};
    clock_gettime(CLOCK_MONOTONIC, &a);
    for (int i = 0; i < N; i++) nanosleep(&d, NULL);
    clock_gettime(CLOCK_MONOTONIC, &b);
    double rel = (b.tv_sec - a.tv_sec) * 1e3 + (b.tv_nsec - a.tv_nsec) / 1e6;
    int64_t span = run_periodic(1000000, N, lat);
    qsort(lat, N, sizeof lat[0], cmp);
    printf("relative sleeps: %d x 1 ms took %.2f ms (drift %.2f ms)\n", N, rel, rel - N);
    printf("absolute sleeps: span %.3f ms for %d periods\n", span / 1e6, N - 1);
    printf("latency us: min %.1f  median %.1f  p99 %.1f  max %.1f\n", lat[0] / 1e3, lat[N / 2] / 1e3, lat[N * 99 / 100] / 1e3, lat[N - 1] / 1e3);
    return 0;
}

In [ ]:
from roboprep import clab
r = clab.sh("gcc -O2 -o compare compare.c periodic.c && ./compare", timeout=60)
print(r.stdout, r.stderr)

## Checkpoint: Math
**M1 (Apply).** Measured latencies range from 4 us to 61 us. Jitter (max minus min) in microseconds?

**M2 (Apply).** A loop uses relative 1 ms sleeps; each pass also costs 60 us of work and wake-up. Drift after
2000 passes, in milliseconds?

**M3 (Apply).** Cortex-M4 interrupt entry takes 12 cycles at 170 MHz. Nanoseconds (1 decimal)?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why does the IMU interrupt handler only read the FIFO count and wake a thread?
- a) handlers cannot call functions
- b) long handlers block other interrupts and the scheduler, raising everyone's latency; the work runs in a thread that can be prioritized and preempted
- c) the FIFO is only readable from threads

**K2 (Evaluate).** List the steps to turn a ROS 2 control node's loop into a real-time thread on Linux and say what
each step protects against.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r03.h`).
3. In `roboprep/checks/r03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()